# Jigsaw Puzzle Reconstruction - Milestone 1 Demonstration

**CSE480 Machine Vision** - classical computer vision only.

This notebook walks the six tasks of the brief, stage by stage, on a real
photograph from the provided dataset and on a synthetic puzzle with a known
answer.  It ends with the end-to-end routine and its quantitative evaluation.

Everything shown is computed by `src/`, which depends on NumPy alone.
Matplotlib is used here only to draw figures, Pillow only to read files.

In [ ]:
import json
import os
import sys
import time

sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

from src import PuzzleSolver
from src import assembly as asm
from src import contour_extraction as ce
from src import edge_detection as ed
from src import edge_matching as em
from src import enhancement as enh
from src import evaluation as ev
from src import segmentation as seg
from src import thresholding as th

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"


def show(images, titles=None, cols=4, size=3.2):
    """Draw a row-major grid of images."""
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    axes = np.atleast_1d(axes).ravel()
    for ax, im in zip(axes, images):
        a = np.asarray(im)
        ax.imshow(a, cmap=None if a.ndim == 3 else "gray")
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    if titles:
        for ax, t in zip(axes, titles):
            ax.set_title(t, fontsize=9)
    plt.tight_layout()
    plt.show()


print("library loaded")

## 0. The input

Two kinds of input are used throughout.

* A **real photograph** from the provided dataset: 35 pieces of one jigsaw
  scattered on a dark cloth.
* A **synthetic puzzle** from `evaluation.generate_puzzle`, which cuts a
  picture into interlocking pieces, shuffles and rotates them, and remembers
  the answer.  Reconstruction accuracy is measurable only because of it.

`main.dataset_scenes()` prefers the full export under `../detection/` and
falls back to the eight photographs committed to `../data/`, so this runs in a
fresh clone with nothing downloaded.

In [ ]:
from PIL import Image

import main as m1


def imread(path, max_side=None):
    img = Image.open(path).convert("RGB")
    if max_side and max(img.size) > max_side:
        s = max_side / max(img.size)
        img = img.resize((int(img.size[0] * s), int(img.size[1] * s)),
                         Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


photo = None
photo_boxes = None
scenes, load_labels = m1.dataset_scenes()
if scenes:
    img_path, lab_path, n_obj = scenes[0]
    photo = imread(img_path, 1920)   # native; DATASET_SOLVER is tuned for it
    photo_boxes = load_labels(lab_path, photo.shape)
    print("photograph:", os.path.basename(img_path),
          f"({len(photo_boxes)} annotated pieces)")
else:
    print("no dataset photographs found; real-photograph cells are skipped")

original = ev.synthetic_source_image(480, 600, seed=114)
scrambled, gt = ev.generate_puzzle(original, rows=4, cols=5, rotate=True,
                                   seed=14)
print("synthetic puzzle:", gt.grid_shape, "pieces, canvas", scrambled.shape)

show([photo if photo is not None else scrambled, original, scrambled],
     ["dataset photograph", "original picture", "scrambled synthetic puzzle"],
     cols=3, size=5)

## 1. Image enhancement

Noise reduction (**mean**, Gaussian and median), contrast adjustment
(histogram equalisation and contrast stretching, both driven by the library's
own histogram) and sharpening (unsharp masking and a Laplacian built on the
shared convolution routine).

The median is the interesting one: it is a *rank* statistic, not a linear
operator, so it has no convolutional or separable form and every output pixel
genuinely needs its own window's order statistics.  The implementation
materialises all windows at once through a stride-trick view so one vectorised
`np.median` does the work in compiled code; the only Python loop left is over
the colour channels.

In [ ]:
crop = (photo if photo is not None else scrambled)[:360, :480]

rng = np.random.default_rng(0)
noisy = np.clip(crop / 255.0 + rng.normal(0, 0.05, crop.shape), 0, 1)
sp = noisy.copy()
sp[rng.random(sp.shape[:2]) < 0.02] = 1.0
sp[rng.random(sp.shape[:2]) < 0.02] = 0.0

show([crop, sp, enh.mean_filter(sp, 3), enh.gaussian_blur(sp, 1.5),
      enh.median_filter(sp, 3)],
     ["original", "salt and pepper", "mean 3x3", "Gaussian sigma=1.5",
      "median 3x3"], cols=5, size=3.0)

for name, img in [("no filtering", sp),
                  ("mean 3", enh.mean_filter(sp, 3)),
                  ("mean 5", enh.mean_filter(sp, 5)),
                  ("gaussian 1.5", enh.gaussian_blur(sp, 1.5)),
                  ("median 3", enh.median_filter(sp, 3)),
                  ("median 5", enh.median_filter(sp, 5))]:
    m = ev.image_metrics(img, crop, allow_rotation=False)
    print(f"{name:14s} PSNR {m['psnr_db']:6.2f} dB   SSIM {m['ssim']:.3f}")

print()
print("The median wins on impulse noise because it is a rank statistic: one")
print("outlying pixel cannot drag the answer, whereas both linear filters")
print("average the salt and pepper back into their neighbourhoods.")

In [ ]:
eq = enh.histogram_equalization(crop)
st = enh.contrast_stretch(crop, 2, 98)

show([crop, eq, st, enh.unsharp_mask(crop, 1.5, 1.2),
      enh.laplacian_sharpen(crop, 0.35)],
     ["original", "histogram equalised", "contrast stretched",
      "unsharp mask", "Laplacian sharpen"], cols=5, size=3.0)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(enh.histogram(crop), label="original")
ax[0].plot(enh.histogram(eq), label="equalised")
ax[0].set_title("histogram")
ax[0].legend()
ax[1].plot(enh.cumulative_histogram(enh.histogram(crop)), label="original")
ax[1].plot(enh.cumulative_histogram(enh.histogram(eq)), label="equalised")
ax[1].set_title("cumulative histogram (the transfer function)")
ax[1].legend()
plt.tight_layout()
plt.show()

## 1d. Thresholding

Global (fixed and iterative/isodata), Otsu, and adaptive (mean via an integral
image, and Gaussian-weighted).

Otsu maximises the between-class variance `w0*w1*(mu0-mu1)^2`.  When the two
modes are well separated that criterion is *flat* across the empty valley
between them, so this implementation returns the **midpoint of the maximising
plateau** rather than its left edge: the same optimum, but as far from both
modes as possible.

In [ ]:
gray = enh.gaussian_blur(crop, 1.0)
t_otsu = th.otsu_threshold(gray)
t_iso = th.isodata_threshold(gray)
print(f"Otsu {t_otsu:.3f}   isodata {t_iso:.3f}")

show([crop,
      th.global_threshold(gray, 0.5),
      th.otsu(gray),
      th.adaptive_threshold(gray, 51, 0.02, "mean"),
      th.adaptive_threshold(gray, 51, 0.02, "gaussian"),
      seg.background_distance(crop)],
     ["input", "global 0.5", f"Otsu {t_otsu:.2f}", "adaptive mean 51",
      "adaptive Gaussian 51", "colour distance to background"],
     cols=3, size=4)

## 2. Edge detection

Sobel and Prewitt return gradient magnitude **and** orientation; Canny adds
non-maximum suppression, double thresholding and hysteresis edge linking.

Defaults are `sigma = 1.4`, `low = 0.05`, `high = 0.15` on the normalised
gradient magnitude (the usual `high ~ 3*low` ratio).  The second cell shows
what each parameter does.

In [ ]:
_, _, sob_mag, sob_dir = ed.sobel(crop)
_, _, pre_mag, pre_dir = ed.prewitt(crop)
edges, stages = ed.canny(crop, sigma=1.4, low=0.05, high=0.15,
                         return_stages=True)


def orientation_rgb(theta, mag):
    """Gradient direction as hue, gradient strength as brightness."""
    a = (np.rad2deg(theta) % 180.0) / 180.0
    rgb = np.stack([np.abs(np.sin(np.pi * (a + k / 3))) for k in range(3)],
                   axis=2)
    return rgb * mag[:, :, None]


show([sob_mag, pre_mag, orientation_rgb(sob_dir, sob_mag),
      orientation_rgb(pre_dir, pre_mag), stages["nms"], stages["strong"],
      edges, np.abs(sob_mag - pre_mag)],
     ["Sobel magnitude", "Prewitt magnitude", "Sobel orientation",
      "Prewitt orientation", "after non-max suppression", "strong edges",
      "Canny (after linking)", "|Sobel - Prewitt|"], cols=4, size=3.6)

print("mean |Sobel - Prewitt| magnitude:",
      f"{np.abs(sob_mag - pre_mag).mean():.4f}")
print("The two agree closely; Sobel's centre-weighted kernel makes it")
print("slightly less sensitive to single-pixel noise.")

In [ ]:
sigmas = [0.8, 1.4, 2.5]
pairs = [(0.02, 0.06), (0.05, 0.15), (0.10, 0.25)]

show([ed.canny(crop, sigma=s, low=0.05, high=0.15) for s in sigmas]
     + [ed.canny(crop, 1.4, lo, hi) for lo, hi in pairs],
     [f"sigma={s}" for s in sigmas]
     + [f"low={lo}, high={hi}" for lo, hi in pairs], cols=3, size=4)

## 3. Segmentation and contour extraction

Foreground mask, then connected components (two-pass union-find over row
runs), then Moore-neighbour boundary tracing with Jacob's stopping criterion,
then one cropped `Piece` per component carrying its mask, contour, centroid
and normalised orientation.

In [ ]:
scene = photo if photo is not None else scrambled
mask = seg.foreground_mask(scene, "background", open_radius=2, close_radius=2)
labels_raw, n_raw = seg.connected_components(mask)
labels, stats = seg.filter_components(labels_raw, min_area_ratio=0.45)
pieces = ce.extract_pieces(scene, labels, stats=stats)
print(f"{n_raw} raw components -> {len(stats)} pieces")

rng = np.random.default_rng(0)
palette = rng.integers(60, 256, size=(labels_raw.max() + 1, 3))
palette[0] = 0
show([scene, mask, palette[labels_raw].astype(np.uint8),
      palette[labels].astype(np.uint8)],
     ["input", "foreground mask", f"{n_raw} raw components",
      f"{len(stats)} kept as pieces"], cols=2, size=6)

In [ ]:
p = max(pieces, key=lambda q: q.area)
outline = np.zeros(p.mask.shape, bool)
outline[p.contour[:, 0], p.contour[:, 1]] = True

show([p.image, p.mask, outline],
     ["cropped piece", "its mask",
      f"traced contour ({len(p.contour)} points)"], cols=3, size=4)

show([ce.normalize_piece(q).image for q in pieces[:8]],
     [f"piece {q.index}" for q in pieces[:8]], cols=4, size=2.4)

## 4. Piece-edge description

Four corners, then four sides, then each side classified `tab` / `blank` /
`flat` from its deviation profile, plus a strip of colour sampled just inside
it.

Corners come from the **body-edge model**: a piece is a rectangle plus bumps,
so its four straight edges share one direction modulo 90 degrees, recovered as
`arg(sum exp(4i*phi))/4` over the tangent angles.  A tab arc sweeps every
direction and cancels itself out of that sum.

In [ ]:
res = PuzzleSolver().solve(scrambled, grid_shape=(4, 5))
descs = res.descriptions
print(f"{len(descs)} pieces described; grid used: {res.grid_shape}")

colours = {"tab": (0, 0.9, 0), "blank": (0, 0.55, 1.0),
           "flat": (1.0, 0.85, 0)}
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, d in zip(axes.ravel(), descs[:12]):
    ax.imshow(d.piece.image)
    for s in d.sides:
        ax.plot(s.points[:, 1], s.points[:, 0], color=colours[s.type], lw=2)
    ax.plot(d.corners[:, 1], d.corners[:, 0], "r.", ms=12)
    ax.set_title(f"piece {d.index}: "
                 + "/".join(s.type[0] for s in d.sides), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

rows, cols = gt.grid_shape
print("expected corner/edge/interior pieces:",
      4, 2 * (rows - 2) + 2 * (cols - 2), (rows - 2) * (cols - 2))
print("observed:",
      sum(1 for d in descs if d.n_flats == 2),
      sum(1 for d in descs if d.n_flats == 1),
      sum(1 for d in descs if d.n_flats == 0))

In [ ]:
d = descs[0]
fig, axes = plt.subplots(2, 4, figsize=(14, 4.5))
for k, s in enumerate(d.sides):
    axes[0, k].plot(s.profile)
    axes[0, k].axhline(0, color="k", lw=0.5)
    axes[0, k].set_ylim(-0.4, 0.4)
    axes[0, k].set_title(f"side {k}: {s.type}", fontsize=9)
    axes[1, k].imshow(np.repeat(s.colors.transpose(1, 0, 2), 10, axis=0))
    axes[1, k].axis("off")
axes[0, 0].set_ylabel("deviation / chord")
plt.tight_layout()
plt.show()

## 5. Piece-edge matching

$$D(a,b) = w_s D_{shape}(a,b) + w_c D_{colour}(a,b) + w_l D_{length}(a,b)$$

with $D=\infty$ for inadmissible pairs (a flat is a puzzle border and can
never be an interior seam; a tab must meet a blank).  When two pieces sit side
by side one side is traversed in the opposite direction to the other and their
outward normals oppose, so a perfect fit satisfies $p_a(t) = -p_b(1-t)$; the
RMS residual of that identity is the shape cost.  The colour strips must
satisfy $C_a(t) = C_b(1-t)$.

Default weights are $w_s = 1$, $w_c = 1$, $w_l = 0.5$.

In [ ]:
table = res.table
assoc = ev.associate_with_ground_truth(res.pieces, gt)
acc = ev.matching_accuracy(table, descs, gt, assoc)
print("weights:", table.weights.as_dict())
print(f"true partner ranked first: {acc['top1_accuracy']:.1%}  "
      f"(top-3 {acc['top3_accuracy']:.1%}, mean rank {acc['mean_rank']:.1f})")
print("best-buddy pairs:", len(em.best_buddies(table)))

step = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "W": (0, -1)}
opp = {"N": "S", "E": "W", "S": "N", "W": "E"}
cell = {}
dirs = {}
for i in range(len(descs)):
    r, c, ang = gt.placements[assoc[i]]
    cell[i] = (int(r), int(c))
    dirs[i] = ev.gt_side_directions(descs[i], ang)
by_cell = {v: k for k, v in cell.items()}

true_s = []
true_c = []
for i in range(len(descs)):
    r, c = cell[i]
    for s in range(4):
        dd = dirs[i][s]
        j = by_cell.get((r + step[dd][0], c + step[dd][1]))
        if j is None:
            continue
        t = dirs[j].index(opp[dd])
        true_s.append(table.shape[i, s, j, t])
        true_c.append(table.colour[i, s, j, t])

fin = np.isfinite(table.cost)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for k, (name, tv, av) in enumerate([("shape", true_s, table.shape[fin]),
                                    ("colour", true_c, table.colour[fin])]):
    ax[k].hist(av, bins=50, density=True, alpha=0.6,
               label="all admissible pairs")
    ax[k].hist(tv, bins=25, density=True, alpha=0.8, label="true seams")
    ax[k].set_title(f"{name}: true {np.mean(tv):.3f} "
                    f"vs all {np.mean(av):.3f}")
    ax[k].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Assembly and the end-to-end routine

Greedy best-first placement on the grid, resolving each piece's **rotation**
as it is placed, with best-buddy seams committed first, a documented
tie-breaking rule, three-stage dead-end relaxation, and restarts from every
candidate corner seed so the best arrangement obtained is always returned.

In [ ]:
a = res.assembly
pid, rot = a.as_arrays()
print("piece grid:")
print(pid)
print("rotation grid (index of the side facing North):")
print(rot)
print("complete:", a.complete, " forced placements:", a.n_forced)
for line in a.log[:6]:
    print(" ", line)

show([scrambled, res.reconstruction, original],
     ["input (scrambled)", "reconstruction", "original picture"],
     cols=3, size=5)

In [ ]:
print(res.summary())
print()
print("reference-free quality")
for k, v in res.quality.items():
    print(f"  {k:24s} {v}")

print()
print("against the ground truth")
print("  position  ", ev.direct_accuracy(a, descs, gt, assoc))
print("  neighbour ", ev.neighbour_accuracy(a, gt, assoc))
print("  rotation  ", ev.rotation_accuracy(a, descs, gt, assoc))
print("  image     ", ev.image_metrics(res.reconstructed_body(), original))

print()
print("timings (seconds)")
for k, v in res.timings.items():
    print(f"  {k:16s} {v:6.2f}")

## How performance scales

Run time is dominated by scoring all $(4N)^2$ side pairs and by re-running the
greedy pass from each candidate seed, so it grows roughly with the square of
the piece count.  Accuracy is unaffected by rotation, which is resolved during
placement, and degrades only when the compatibility measure itself becomes
ambiguous.

In [ ]:
rows_out = []
for (r, c, seed) in [(2, 3, 11), (3, 4, 12), (4, 5, 14), (5, 7, 16)]:
    src = ev.synthetic_source_image(110 * r, 110 * c, seed=100 + seed)
    scr, g = ev.generate_puzzle(src, rows=r, cols=c, rotate=True, seed=seed)
    t0 = time.perf_counter()
    out = PuzzleSolver().solve(scr, (r, c))
    dt = time.perf_counter() - t0
    at = ev.associate_with_ground_truth(out.pieces, g)
    nbr = ev.neighbour_accuracy(out.assembly, g, at)["neighbour_accuracy"]
    pos = ev.direct_accuracy(out.assembly, out.descriptions, g,
                             at)["position_accuracy"]
    rows_out.append((f"{r}x{c}", r * c, dt, nbr, pos))
    print(f"{r}x{c}  {r * c:3d} pieces  {dt:6.2f}s  "
          f"neighbour {nbr:.2f}  position {pos:.2f}")

n = [q[1] for q in rows_out]
t = [q[2] for q in rows_out]
plt.figure(figsize=(5, 3.2))
plt.plot(n, t, "o-")
plt.xlabel("pieces")
plt.ylabel("seconds")
plt.title("run time vs puzzle size")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## The real photographs

The dataset photographs are the actual target of this milestone.  Three things
make them harder than a synthetic cut:

* several pieces physically **touch**, and two touching pieces are one
  connected component -- no threshold separates them, so a distance-transform
  watershed (`segmentation.split_touching`) is needed;
* the pieces lie all over a table under uneven light, so colours across a true
  seam disagree even where the picture continues -- hence
  `colour_norm="meanstd"` and the gradient-based `colour_metric="mgc"`;
* the printed picture is **almost entirely white**.  Measured on-piece over
  2.5 M pixels: mean chroma 0.062, with 90 % of the printed area below 0.10.
  The photometric term therefore has very little to work with.

The dataset also carries a hidden answer key: the class ids are the row-major
positions of the finished 5x7 puzzle, so reconstruction can be **scored** on
real photographs and not only on synthetic ones.

In [ ]:
if photo is not None:
    # main.DATASET_SOLVER.  The two settings that differ from the library
    # defaults are colour_metric="mgc" (an illumination-invariant photometric
    # term) and border_mode="soft" (the flat/tab/blank classification is right
    # on only about 80 % of pieces here, so the border rule is charged as a
    # penalty rather than enforced as a filter).
    out = PuzzleSolver(open_radius=2, close_radius=2, min_area_ratio=0.45,
                       max_area_ratio=1.7, colour_norm="meanstd",
                       colour_metric="mgc",
                       border_mode="soft").solve(photo, (5, 7))
    print(out.summary())
    for note in out.notes:
        print(" ", note)
    print("flat sides found:",
          sum(1 for d in out.descriptions for s in d.sides if s.is_flat),
          "of 24")
    print("corner pieces:",
          sum(1 for d in out.descriptions if d.is_corner_piece), "of 4")

    cells = m1.dataset_true_cells(out.pieces, photo_boxes, photo.shape)
    nbr = ev.neighbour_accuracy_from_cells(
        out.assembly, cells)["neighbour_accuracy"]
    pos = ev.position_accuracy_from_cells(
        out.assembly, cells)["position_accuracy"]
    print()
    print(f"  neighbour accuracy {nbr:.3f}   (about 0.05 by chance)")
    print(f"  position accuracy  {pos:.3f}   (about 0.03 by chance)")
    print()
    print("Over all 50 photographs the study reports 0.220 neighbour and")
    print("0.131 position accuracy; a single photograph varies either side")
    print("of that.  See results/evaluation_results/dataset_study.json.")

    show([photo, out.mask, out.reconstruction],
         ["photograph", "foreground mask",
          f"reconstruction -- position accuracy {pos:.2f}"], cols=3, size=5)

    print()
    print("READ THE NUMBERS, NOT THE PICTURE.  The right-hand panel is a")
    print("clean 5x7 rectangle of real pieces and looks like a solved")
    print(f"puzzle, but only {pos:.0%} of the pieces are in the right cell")
    print(f"({round(pos * 35)} of 35).  The assembler fills every cell by")
    print("construction, so a tiled rectangle is evidence about the")
    print("segmentation and the renderer, not about the matching.")
    print("Report section 9.3 explains why.")
else:
    print("dataset not available in this checkout")

## What to take from this notebook

1. Every operation the brief asks for is implemented from scratch and shown
   working above, on real input.
2. Segmentation **succeeds** on the dataset photographs: 0.973 piece recall at
   0.963 precision over all 50.  Description is close but not perfect --
   averaged over those 50 the library finds 23.1 flat sides against the 24 a
   5x7 grid must expose, and 3.2 corner pieces of 4; on a good photograph it
   finds exactly 24 and exactly 4, on a poor one fewer.
3. Reconstruction from seam matching **fails** on them, because this jigsaw's
   picture is almost entirely white and a seam therefore carries almost no
   information.  The matcher runs at top-1 0.32 against 0.10 for chance, where
   the same measure reaches 0.90 on synthetic puzzles.
4. On synthetic puzzles, where the picture carries content, the identical code
   reconstructs **perfectly** -- which places the limitation in the input
   rather than in the method.
5. A reconstructed image that looks like a neat rectangle is not evidence of
   anything.  Only the scored numbers say whether it worked.

---

# Milestone 2 - Learned compatibility

Milestone 1 ends where the section above ends: the pieces are found and
described correctly, and the real puzzle is not solved.  Milestone 2 replaces
the hand-designed compatibility measure with two **learned** ones and asks
whether that closes the gap.

The comparison is deliberately narrow.  Segmentation, contour tracing, corner
location, side description and **the assembly algorithm itself** are unchanged
from Milestone 1.  The only thing that differs is the function answering
*"how well do these two sides fit?"*, and all three feed the same
`CompatibilityTable` interface, so any difference is attributable to the
matcher.

```
segment -> describe -> | classical: shape + colour formula |
                       | Siamese CNN                       | -> same assembler
                       | graph neural network              |
```

Everything below reads the checkpoints and result files that
`python main_milestone2.py --scaling --hard --figures` writes into
`../results/milestone2/`.

In [ ]:
import torch

from src.ml.features import VECTOR_DIM
from src.ml.gnn import PuzzleGNN
from src.ml.infer import gnn_table, predicted_matches, siamese_table
from src.ml.siamese import SiameseCNN

MDIR = os.path.abspath(os.path.join("..", "results", "milestone2"))


def load_json(name):
    with open(os.path.join(MDIR, name), encoding="utf-8") as fh:
        return json.load(fh)


siamese = SiameseCNN()
siamese.load_state_dict(torch.load(os.path.join(MDIR, "siamese_cnn.pt"),
                                   map_location="cpu"))
siamese.eval()

gnn = PuzzleGNN(VECTOR_DIM)
gnn.load_state_dict(torch.load(os.path.join(MDIR, "graph_nn.pt"),
                               map_location="cpu"))
gnn.eval()

for name, m in [("Siamese CNN", siamese), ("graph network", gnn)]:
    k = sum(q.numel() for q in m.parameters())
    print(f"{name:14s} {k:>9,} parameters  ({k * 4 / 1e6:.2f} MB)")

## Task 1 - Dataset preparation

Labels come from the synthetic generator, which knows every true seam by
construction.  Pairs are split **by puzzle, never by pair**, so no piece of a
test puzzle is seen in training, and the same split is shared by both models.
Negatives are mined at a fixed ratio per positive; augmentation perturbs
illumination, contrast, colour balance and noise.

In [ ]:
info = load_json("dataset.json")
print("puzzles:", info["n_puzzles"], " split:", info["split_puzzles"])
print("sizes:", info["sizes"])
print("negatives per positive:", info["negatives_per_positive"])
print("augmentation:", ", ".join(info["augmentation"]))
print()
for split, bal in info["pairs"].items():
    print(f"  {split:5s} {bal['n_pairs']:6d} pairs  "
          f"{bal['positive']:5d} positive / {bal['negative']:6d} negative "
          f"({bal['positive_fraction']:.1%} positive)")
print()
print(info["split_note"])

## Tasks 2 and 3 - The two models, and their training

**Siamese CNN.** A shared-weight encoder embeds each side's colour strip, and
a symmetric head scores the pair.  It sees the full strip, so it has the
richer input.

**Graph neural network.** Each side becomes a node carrying a compact
descriptor; message passing lets a candidate be judged *against its rivals*
rather than in isolation.  It is the smaller and much cheaper model.

Both are checked for over- and under-fitting by the train/validation AUC gap
at the selected epoch.

In [ ]:
tr = load_json("training.json")
print("hyper-parameters:", tr["config"])
print()
for key, label in [("siamese_cnn", "Siamese CNN"), ("graph_nn", "graph network")]:
    d = tr[key]
    h = d["history"]
    print(f"{label}")
    print(f"  parameters {d['parameters']:>9,}   size {d['size_mb']:.2f} MB")
    print(f"  best val AUC {h['best_val_auc']:.4f} at epoch {h['best_epoch']}")
    print(f"  training time {h['training_seconds']:.0f} s")
    print(f"  diagnosis: {d['diagnosis']}")
    print()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for key, label in [("siamese_cnn", "Siamese CNN"), ("graph_nn", "graph network")]:
    h = tr[key]["history"]
    ax[0].plot(h["train_loss"], label=f"{label} train")
    ax[0].plot(h["val_loss"], "--", label=f"{label} val")
    ax[1].plot(h["train_auc"], label=f"{label} train")
    ax[1].plot(h["val_auc"], "--", label=f"{label} val")
ax[0].set_title("loss")
ax[0].set_xlabel("epoch")
ax[0].legend(fontsize=8)
ax[1].set_title("AUC")
ax[1].set_xlabel("epoch")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Neither model over-fits: the train/validation AUC gaps at the selected")
print("epoch are ~0.00.  Validation AUC sits above training AUC early on")
print("because the training batches are augmented and half their negatives")
print("are mined from the hardest candidates, while validation is neither.")

## Task 4 - Assembly, and the four required per-pair outputs

Each matcher produces a `CompatibilityTable`; the **same** greedy assembler of
Milestone 1 consumes it.  For every side the brief asks a model for four
things: whether the two sides are neighbours, the numerical score, which sides
matched, and their relative orientation.  `predicted_matches` returns exactly
that.

In [ ]:
demo_src = ev.synthetic_source_image(480, 600, seed=207)
demo_scr, demo_gt = ev.generate_puzzle(demo_src, rows=4, cols=5, rotate=True,
                                       seed=207)
demo = PuzzleSolver().solve(demo_scr, (4, 5))
dd = demo.descriptions

tables = {"classical": demo.table,
          "Siamese CNN": siamese_table(siamese, dd),
          "graph network": gnn_table(gnn, dd, demo.table)}

print("the four required outputs, first three sides, per method")
for name, tab in tables.items():
    print(f"\n{name}")
    for row in predicted_matches(tab, dd)[:3]:
        print("  ", row)

In [ ]:
demo_assoc = ev.associate_with_ground_truth(demo.pieces, demo_gt)
panels, titles = [], []
for name, tab in tables.items():
    a2 = asm.assemble(dd, tab, (4, 5))
    nbr = ev.neighbour_accuracy(a2, demo_gt, demo_assoc)["neighbour_accuracy"]
    pos = ev.direct_accuracy(a2, dd, demo_gt, demo_assoc)["position_accuracy"]
    panels.append(asm.render_assembly(dd, a2))
    titles.append(f"{name}\nneighbour {nbr:.2f}, position {pos:.2f}")
    print(f"{name:14s} neighbour {nbr:.3f}   position {pos:.3f}")

show(panels, titles, cols=3, size=5)

## Task 5 - Performance evaluation

Four conditions, hardest last: the standard unseen test split, the same split
grouped by puzzle size, a controlled sweep that fades the picture's texture
out, and finally the real dataset photographs.

In [ ]:
cmp = load_json("comparison.json")
names = ["classical", "siamese_cnn", "graph_nn"]
labels = {"classical": "classical", "siamese_cnn": "Siamese CNN",
          "graph_nn": "graph network"}

print("STANDARD TEST SPLIT (unseen synthetic puzzles)")
print(f"{'':28s}" + "".join(f"{labels[m]:>16s}" for m in names))
for sect, key, lab in [("matching", "top1", "matching top-1"),
                       ("matching", "auc", "matching AUC"),
                       ("reconstruction", "neighbour_accuracy", "neighbour acc."),
                       ("reconstruction", "position_accuracy", "position acc."),
                       ("reconstruction", "orientation_accuracy", "orientation acc."),
                       ("reconstruction", "complete_reconstructions", "complete recons."),
                       ("cost", "seconds_match_per_puzzle", "seconds / puzzle"),
                       ("cost", "parameters", "parameters")]:
    row = f"  {lab:26s}"
    for m in names:
        v = cmp[m][sect][key]
        row += f"{v:>16.4f}" if isinstance(v, float) else f"{v:>16,}"
    print(row)

print()
print("All three sit within 0.02 of each other on neighbour accuracy and")
print("reconstruct the same number of puzzles completely: the split is")
print("saturated, so it cannot separate the methods.  The one puzzle that")
print("fails, fails for every method -- which points at its segmentation")
print("rather than at any matcher.")

In [ ]:
sc = load_json("scaling.json")
print("BY PUZZLE SIZE  (AUC / neighbour accuracy)")
print(f"{'grid':>6s}" + "".join(f"{labels[m]:>22s}" for m in names))
for size, d in sc.items():
    row = f"{size:>6s}"
    for m in names:
        row += f"{d[m]['auc']:>13.3f} /{d[m]['neighbour_accuracy']:>7.2f}"
    print(row)
print()
print("Size is what finally separates them.  At 3x4 and 4x5 every method is")
print("perfect; at 5x7 -- the size of the real jigsaw -- the ceiling breaks")
print("and an ordering appears.")

sw = load_json("texture_sweep.json")
xs = [float(k.split("_")[1]) for k in sw]
plt.figure(figsize=(6, 3.6))
for m in names:
    plt.plot(xs, [sw[k][m]["neighbour_accuracy"] for k in sw], "o-",
             label=labels[m])
plt.gca().invert_xaxis()
plt.xlabel("texture retained  (1.0 = normal picture, 0 = flat grey)")
plt.ylabel("neighbour accuracy")
plt.title("What happens as the picture stops helping")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

for k in sw:
    print(f"  texture {k.split('_')[1]:>4s}  " + "   ".join(
        f"{labels[m]} {sw[k][m]['neighbour_accuracy']:.2f}" for m in names))
print()
print("This sweep uses twenty puzzles per level.  An earlier version ran four")
print("and concluded the Siamese was uniquely robust to a fading picture; at")
print("twenty that reverses, and the classical measure is the most robust of")
print("the three.  With four puzzles one reconstruction is 25 % of the score,")
print("so the earlier table was reporting sampling noise as a finding.")

In [ ]:
real = load_json("real_photographs.json")
print("THE REAL DATASET PHOTOGRAPHS  (a genuine domain shift:")
print("both models were trained on generated puzzles)")
print()
for m in names:
    v = real[m]
    print(f"  {labels[m]:14s} neighbour {v['neighbour_accuracy']:.3f}   "
          f"position {v['position_accuracy']:.3f}   "
          f"({v['n_images']} images)")
print()
print("None of the three reconstructs the real puzzle and none is")
print("meaningfully ahead.  Classical and Siamese differ by 0.010 of")
print("neighbour accuracy, far inside the spread across the eight images,")
print("and on position accuracy the classical measure is the best of the")
print("three.  Learning the compatibility measure does not help here: a seam")
print("on this puzzle is white on both sides, and no model separates two")
print("white strips that belong together from two that do not.")
print()
print("The reconstruction-quality column of comparison.json must NOT be read")
print("across methods -- it normalises by each method's own cost table, and")
print("the learned tables are -log p while the classical one is a weighted")
print("RMS distance, so the three are in different units.")

## Milestone 2 conclusions

1. **A saturated benchmark hides everything.**  All three methods reconstruct
   8 of 9 standard test puzzles.  Only the 5x7 slice separates them.
2. **Sample size decided a headline claim, and nearly got it wrong.**  The
   texture sweep at four puzzles per level said learning buys robustness; at
   twenty it says the opposite.
3. **Learning did not buy robustness to a vanishing picture.**  Where the
   picture is informative the hand-designed formula is already at the ceiling;
   where it fades, the classical measure degrades most gracefully; and on the
   real photographs none of the three separates.
4. **Representation beat capacity.**  The Siamese wins on accuracy at 5x7, but
   through a richer input (the full strip versus pooled colour bins), not
   through its extra parameters -- the graph model reaches nearly the same AUC
   on 3.5x fewer parameters and 57x less training time.
5. **Compare against the baseline your own project runs.**  The classical
   baseline here uses the full `main.DATASET_SOLVER` settings and the same
   `border_mode="soft"` assembly as the learned methods.  Getting that wrong
   silently flatters the learned models.
6. **The binding constraint is the training distribution**, not the model.
   The next step is training data drawn from the photographs themselves, which
   needs side-level labels the dataset does not carry.